# loop 循環執行,直到滿足


In [8]:
#0-9
for i in range(10):
    print(i+1)

1
2
3
4
5
6
7
8
9
10


In [3]:
list(range(10))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [11]:
sum=0
for i in range(10):
    sum=sum+(i+1)
print(sum)

55


In [16]:
for i in range(10):
    print("&"*(i+1))


&
&&
&&&
&&&&
&&&&&
&&&&&&
&&&&&&&
&&&&&&&&
&&&&&&&&&
&&&&&&&&&&


In [31]:
for i in range(10):
    print(f'1*{(i+1)}={1*(i+1)}')

1*1=1
1*2=2
1*3=3
1*4=4
1*5=5
1*6=6
1*7=7
1*8=8
1*9=9
1*10=10


In [40]:
print(list(range(2,102,2)))

[2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100]


#

# 九九乘法
1\*1=1


In [13]:

for j in range(1,6):
    print(f'1*{j}={1*j}',end=",")
print()
for j in range(1,6):
    print(f'2*{j}={2*j}',end=",")
print()

1*1=1,1*2=2,1*3=3,1*4=4,1*5=5,
2*1=2,2*2=4,2*3=6,2*4=8,2*5=10,


In [17]:
for i in range(1,6):
    for j in range(1,6):
        print(f'{i}*{j}={i*j}',end=",")
        print()

1*1=1,
1*2=2,
1*3=3,
1*4=4,
1*5=5,
2*1=2,
2*2=4,
2*3=6,
2*4=8,
2*5=10,
3*1=3,
3*2=6,
3*3=9,
3*4=12,
3*5=15,
4*1=4,
4*2=8,
4*3=12,
4*4=16,
4*5=20,
5*1=5,
5*2=10,
5*3=15,
5*4=20,
5*5=25,


In [ ]:
from functools import lru_cache
from math import isqrt
from time import perf_counter

@lru_cache(maxsize=4096)
def check_prime(number):
    """回傳（是否為質數、找到的因數），使用精確的整數運算。"""
    if number < 2:
        return False, None
    if number in (2, 3):
        return True, None
    for divisor in (2, 3):
        if number % divisor == 0:
            return False, divisor
    # 大於 3 的質數只可能是 6k - 1 或 6k + 1，檢查到平方根即可。
    limit = isqrt(number)
    for divisor in range(5, limit + 1, 6):
        if number % divisor == 0:
            return False, divisor
        if divisor + 2 <= limit and number % (divisor + 2) == 0:
            return False, divisor + 2
    return True, None

def parse_prime_query(text):
    """接受單一整數、多個整數，或包含兩端的 a..b 區間。"""
    if ".." in text:
        start_text, end_text = text.split("..")
        start, end = int(start_text), int(end_text)
        if start > end:
            raise ValueError("區間起點不能大於終點。")
        return True, range(start, end + 1)
    parts = text.replace("，", " ").replace(",", " ").split()
    if not parts:
        raise ValueError("請輸入至少一個整數。")
    return False, [int(part) for part in parts]

def describe_prime(number, prime, factor):
    if prime:
        return f"{number} 是質數。"
    if number < 2:
        return f"{number} 不是質數：質數必須是大於 1 的整數。"
    return f"{number} 不是質數：{number} = {factor} × {number // factor}。"

def run_prime_checker():
    history = []
    checked_count = 0
    prime_count = 0
    check_prime.cache_clear()
    help_text = (
        "單筆：7｜批次：7, 12, 23（也可用空白或中文逗號）\n"
        "區間：1..100（列出兩端之間的質數）\n"
        "指令：help 說明｜history 歷史｜stats 統計｜clear 清除歷史｜q 結束"
    )
    print("質數查詢工具\n" + help_text)
    while True:
        try:
            text = input("請輸入數字或指令：").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n已結束質數查詢。")
            break
        command = text.lower()
        if command in ("q", "quit", "exit"):
            print("已結束質數查詢。")
            break
        if command == "help":
            print(help_text)
            continue
        if command == "history":
            print("\n".join(history) if history else "目前沒有查詢歷史。")
            continue
        if command == "clear":
            history.clear()
            print("已清除查詢歷史。")
            continue
        if command == "stats":
            cache = check_prime.cache_info()
            print(f"累計判斷 {checked_count} 個數字：質數 {prime_count} 個，非質數 {checked_count - prime_count} 個（含重複查詢）。")
            print(f"快取命中 {cache.hits} 次，目前儲存 {cache.currsize} 筆結果。")
            continue
        try:
            is_range, numbers = parse_prime_query(text)
        except ValueError:
            print("輸入無效：請輸入整數、多個整數，或起點不大於終點的 a..b 區間。")
            continue
        try:
            started = perf_counter()
            results = [(number, *check_prime(number)) for number in numbers]
            primes = [number for number, prime, factor in results if prime]
            elapsed_ms = (perf_counter() - started) * 1000
            if is_range:
                output = f"區間內共有 {len(primes)} 個質數：" + (", ".join(map(str, primes)) or "無")
            else:
                output = "\n".join(describe_prime(*result) for result in results)
        except KeyboardInterrupt:
            print("\n已取消本次查詢，可以繼續輸入。")
            continue
        checked_count += len(results)
        prime_count += len(primes)
        print(output)
        print(f"判斷耗時：{elapsed_ms:.3f} 毫秒")
        history.append(f"{text} → {output}（{elapsed_ms:.3f} 毫秒）")

run_prime_checker()

In [ ]:
# 費波那契數列：列出初始狀態及 90 次迴圈的 a、b
a, b = 1, 1

print("迴圈次數\ta\tb")
print(f"0（初始）\t{a}\t{b}")

for i in range(1, 91):
    a, b = b, a + b
    print(f"{i}\t{a}\t{b}")